## Statistical Tests

The following section contains various Statistical Tests (p-value tests). It aims to provide more insights and basis of comparison for the various models against each other.

The following tests are present:

- Cochran's Q test for **comparing all 5 models at once** across one overall test to determine whether there is a statistically significant difference in **all models'** performance
- McNemar's test for comparing **two models at a time** as an extension to determine statistically significant difference in performance **between 2 models** on the same test

### Statistical Assumption
All statistical tests are conducted on paired binary correctness (correct vs incorrect predictions). This ensures that the assumption of dependent samples required for both Cochran’s Q test and McNemar’s test is satisfied.


### Conversion of predictions to correctness

Both tests are run on correctness (whether each prediction was correct). Hence, we create both an array of correct predictions and also create a correctness table, where for each row a 1 indicates a correct prediction and a 0 indicates an incorrect prediction.

In [ ]:
import pandas as pd
import numpy as np

y_true = np.array(y_test)
correct_df = pd.DataFrame({
    name: (np.array(pred) == y_true).astype(int)
    for name, pred in predictions.items()
})
print("Correctness table for first test sample:")
correct_df.head()

Correctness table for first test sample:


,NB,LR,SVM,RF,BiLSTM
0,0,0,1,0,1
1,1,1,1,1,1
2,1,1,1,1,1
3,1,1,1,1,1
4,1,1,1,1,1


### Cochran's Q test

Cochran’s Q test is used to determine whether there is an overall statistically significant difference in prediction performance across all five models on the same test set.

Null hypothesis ($H_0$): All five models have the same prediction performance on the same test set.

Alternative hypothesis ($H_1$): At least one model has a different prediction performance from the others on the same test set.

In [ ]:
from statsmodels.stats.contingency_tables import cochrans_q

q_result = cochrans_q(correct_df)
print("Cochran's Q statistic:", q_result.statistic)
print("p-value:", q_result.pvalue)

Cochran's Q statistic: 1334.8773200543233
p-value: 9.122933333614289e-288


**Observations on Cochran's Q test:**

With the extremely small p-value obtained, we reject the null hypothesis and conclude that there is a statistically significant difference in performance among the five models evaluated on the same test set.

However, `Cochran’s Q` test does not indicate which specific models differ from one another. To identify where these differences lie, we will use `McNemar’s test` for pairwise comparisons.

### McNemar's test

McNemar’s test is used to compare pairs of models on the same test set and determine whether their differences in prediction performance were statistically significant. Due to the large number of pair-wise comparisons, Bonferroni correction was applied to adjust for these multiple pair-wise comparisons and reduce the risk of false positives.

For each pair-wise comparison, we have the following hypotheses:

Null hypothesis (**$H_0$**): The two models have the same prediction performance on the same test set.

Alternative hypothesis (**$H_1$**): The two models have different prediction performance on the same test set.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from itertools import combinations

def mcnemar_pair(y_true, pred1, pred2):
    pred1 = np.array(pred1)
    pred2 = np.array(pred2)
    y_true = np.array(y_true)

    correct1 = pred1 == y_true
    correct2 = pred2 == y_true

    b = np.sum((correct1 == 1) & (correct2 == 0))
    c = np.sum((correct1 == 0) & (correct2 == 1))

    table = [[0, b],
             [c, 0]]

    result = mcnemar(table, exact=True)
    return result.pvalue, b, c

pairwise_results = []
for m1, m2 in combinations(predictions.keys(), 2):
    p, b, c = mcnemar_pair(y_true, predictions[m1], predictions[m2])
    pairwise_results.append({
        "Model_1": m1,
        "Model_2": m2,
        "b": b,
        "c": c,
        "p_value": p
    })

pairwise_df = pd.DataFrame(pairwise_results)
pairwise_df["p_bonferroni"] = multipletests(pairwise_df["p_value"], method="bonferroni")[1]
pairwise_df["significant_0.05"] = pairwise_df["p_bonferroni"] < 0.05

pairwise_df = pairwise_df.sort_values(["Model_1", "Model_2"]).reset_index(drop=True)

models_list = list(predictions.keys())
p_matrix = pd.DataFrame("-", index=models_list, columns=models_list)

for _, row in pairwise_df.iterrows():
    m1, m2 = row["Model_1"], row["Model_2"]
    pval = row["p_bonferroni"]
    p_matrix.loc[m1, m2] = f"{pval:.4g}"
    p_matrix.loc[m2, m1] = f"{pval:.4g}"

print(p_matrix)

                NB         LR         SVM          RF      BiLSTM
NB               -  2.75e-161  5.445e-143  1.427e-127  6.023e-126
LR       2.75e-161          -           1       0.862      0.2242
SVM     5.445e-143          1           -           1      0.5968
RF      1.427e-127      0.862           1           -           1
BiLSTM  6.023e-126     0.2242      0.5968           1           -


**Observations from McNemar's test**

After applying `Bonferroni` correction, McNemar's test shows that all pairwise comparisons involving Naive Bayes remain **statistically significant**. This indicates that it performs significantly worse than the other models.

In contrast, there are no statistically significant differences that is observed among Logistic Regression, SVM, Random Forest, and Bi-LSTM, as their adjusted p-values exceed the 0.05 significance level. This suggests that these stronger models achieve similar levels of performance on the test set.

### Effect Size

The effect size analysis provides insight into the practical magnitude of performance differences between the models.

In [ ]:
# To show absolute accuracy differences
# Whether statistically significant results are also practically meaningful.

accuracy_summary = correct_df.mean().sort_values(ascending=False) * 100
accuracy_summary = accuracy_summary.round(2)

print("Accuracy by model ranked (%):")
print(accuracy_summary)

effect_size_results = []
for m1, m2 in combinations(predictions.keys(), 2):
    acc1 = correct_df[m1].mean()
    acc2 = correct_df[m2].mean()
    
    effect_size_results.append({
        "Model_1": m1,
        "Model_2": m2,
        "Accuracy_1 (%)": round(acc1 * 100, 2),
        "Accuracy_2 (%)": round(acc2 * 100, 2),
        "Absolute accuracy difference (pp)": round(abs((acc1 - acc2) * 100), 2),
        "Better model": m1 if acc1 > acc2 else (m2 if acc2 > acc1 else "Tie")
    })

effect_size_df = pd.DataFrame(effect_size_results)
effect_size_df = effect_size_df.sort_values(
    by="Absolute accuracy difference (pp)",
    ascending=False
).reset_index(drop=True)

print("\nPairwise effect size summary:")
display(effect_size_df)

Accuracy by model ranked (%):
LR        81.51
SVM       81.36
RF        81.02
BiLSTM    80.86
NB        72.39
dtype: float64

Pairwise effect size summary:


,Model_1,Model_2,Accuracy_1 (%),Accuracy_2 (%),Absolute accuracy difference (pp),Better model
0,NB,LR,72.39,81.51,9.13,LR
1,NB,SVM,72.39,81.36,8.97,SVM
2,NB,RF,72.39,81.02,8.63,RF
3,NB,BiLSTM,72.39,80.86,8.47,BiLSTM
4,LR,BiLSTM,81.51,80.86,0.65,LR
5,SVM,BiLSTM,81.36,80.86,0.50,SVM
6,LR,RF,81.51,81.02,0.49,LR
7,SVM,RF,81.36,81.02,0.34,SVM
8,RF,BiLSTM,81.02,80.86,0.16,RF
9,LR,SVM,81.51,81.36,0.15,LR


**Effect size interpretation**

Naive Bayes shows a substantial performance gap compared to all other models, with accuracy differences ranging betwen approximately 8.5 to 9.1 percentage points. This indicates that its poorer performance is not only **statistically significant**, but also **practically meaningful**.

In contrast, the differences among Logistic Regression, SVM, Random Forest, and BiLSTM are all below 1 percentage point. Such small differences are negligible in practical terms and helps to explain why McNemar’s test did not detect statistically significant differences between these models after correction. This reinforces that these differences are negligible in practical terms.

Overall, this suggests that while Naive Bayes is clearly a weaker model for this task, the stronger models perform at a very similar level. The remaining errors are therefore more likely driven by inherent task difficulty, such as class ambiguity and overlap, rather than substantial differences in modelling capability.

---

### Statistical tests summary

Considering all 3 tests, they present a consistent picture of model performance. 

- `Cochran’s Q test` confirms that there are statistically significant differences among the models. Whereas `McNemar’s test` shows that these differences are primarily driven by Naive Bayes, which performs significantly worse than all other models. Lastly, `Effect size analysis` further demonstrates that this gap is substantial, with accuracy differences of approximately 8.5 to 9.1 percentage points, indicating a practically meaningful difference.

- In contrast, Logistic Regression, SVM, Random Forest, and Bi-LSTM show neither statistically significant nor practically meaningful differences in performance. Their accuracy differences are below 1 percentage point. This suggests that these stronger models operate at a similar performance level on the task.

- To conclude, the poorer performance of Naive Bayes can be attributed to its reliance on its independence assumptions and surface level lexical features. This makes it less effective for handling complex or ambiguous inputs. Meanwhile, the comparable performance among the stronger models indicates that remaining errors are more likely due to inherent task difficulty like class ambiguity and overlap rather than limitations of the model.

### Alignment with structured error analysis

The statistical findings from the statistical analysis are consistent with the structured error analysis conducted earlier. In particular, Naive Bayes was observed to rely heavily on surface level lexical cues and exhibited systematic misclassification patterns, such as overpredicting certain classes. This explains the large and statistically significant performance gap observed in both McNemar’s test and the effect size analysis.

For the stronger models, the error analysis showed that misclassifications are concentrated on inherently ambiguous cases, especially between `not_cyberbullying` and `other_cyberbullying`. This aligns with the statistical results, where differences among these models are both statistically insignificant and practically negligible, indicating that performance is constrained more by dataset ambiguity than by model capability.

### Limitation of Statistical Analysis

However, it is also important to note that the statistical tests are based on binary correctness (correct vs incorrect predictions). This does not capture class specific performance differences. In particular, this may mask variations in precision and recall across different categories, especially in a multi-class setting where some classes are inherently more difficult than others.